# Colab Fine-Tuning HDemucs on Sanidha 2-Stem

Fine-tunes pretrained HDemucs (4-source) to 2-stem (vocals vs accompaniment) on Sanidha Carnatic dataset.

In [ ]:
# Cell 1: Environment Setup & Drive Mount
!nvidia-smi

from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/Samar-Talwar/Computational-Musicology-for-Indian-Art-Music.git repo || (cd repo && git pull origin main)

!pip install -q torchaudio demucs-infer soundfile pandas auraloss

import sys
sys.path.append('/content/repo')

print("Environment ready")

In [ ]:
# Cell 2: Dataset Extraction to Fast NVMe
import os
import zipfile
from pathlib import Path

zip_path = Path('/content/drive/MyDrive/sanidha_train_2stem.zip')
extract_dir = Path('/content/data/staged_train_2stem')
extract_dir.mkdir(parents=True, exist_ok=True)

if not zip_path.exists():
    raise FileNotFoundError(f"Dataset zip not found at {zip_path}")

print(f"Extracting {zip_path} -> {extract_dir}")
with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(extract_dir)

# Verify extraction
manifest = extract_dir / 'manifest.csv'
assert manifest.exists(), f"Manifest not found at {manifest}"

import pandas as pd
df = pd.read_csv(manifest)
print(f"Manifest rows: {len(df)}")
print(f"Columns: {list(df.columns)}")

wav_count = len(list(extract_dir.rglob('*.wav')))
print(f"WAV files extracted: {wav_count}")
assert wav_count == 10242, f"Expected 10242 WAV files, got {wav_count}"

print("Dataset extraction verified")

In [ ]:
# Cell 3: DataLoader Configuration
from models.dataset import SanidhaStagedDataset
from torch.utils.data import DataLoader

manifest_path = '/content/data/staged_train_2stem/manifest.csv'
dataset = SanidhaStagedDataset(manifest_path)
print(f"Dataset size: {len(dataset)}")

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    drop_last=True
)

# Batch test
mix_batch, targets_batch = next(iter(dataloader))
print(f"Mix batch shape: {mix_batch.shape}")           # (B, 2, 264600)
print(f"Targets batch shape: {targets_batch.shape}")   # (B, 2, 2, 264600)
assert mix_batch.shape == (4, 2, 264600)
assert targets_batch.shape == (4, 2, 2, 264600)
print("DataLoader verified")

In [ ]:
# Cell 4: Model Architecture & Adaptation
import torch
import torch.nn as nn
from demucs_infer.pretrained import get_model as get_hdemucs
from demucs_infer.apply import apply_model

# Load pretrained HDemucs (4-source)
model = get_hdemucs('demucs_unittest')  # HDemucs, 2 ch, 44100 Hz, 4 stems
print(f"Original sources: {model.sources}")
print(f"Audio channels: {model.audio_channels}")
print(f"Sample rate: {model.samplerate}")

# Adapt output projection: 4 sources -> 2 sources (vocals, accompaniment)
# We keep encoder frozen, retrain decoder heads for 2 sources
for param in model.encoder.parameters():
    param.requires_grad = False

# Replace decoder output layer to produce 2 sources instead of 4
# The decoder output is typically a Conv1d with out_channels = 4 * audio_channels
# We'll modify the final layer
old_out_channels = model.decoder[-1].out_channels
new_out_channels = 2 * model.audio_channels  # 2 sources x 2 channels

model.decoder[-1] = nn.Conv1d(
    in_channels=model.decoder[-1].in_channels,
    out_channels=new_out_channels,
    kernel_size=model.decoder[-1].kernel_size,
    stride=model.decoder[-1].stride,
    padding=model.decoder[-1].padding,
    bias=model.decoder[-1].bias is not None
)

model.sources = ['vocals', 'accompaniment']
print(f"Adapted sources: {model.sources}")
print(f"Decoder output channels: {new_out_channels}")

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)
print(f"Model on {device}")

# Quick forward test
x = torch.randn(1, 2, 264600, device=device)
with torch.no_grad():
    out = model(x)
print(f"Forward output shape: {out.shape}")  # (1, 2, 2, 264600)

In [ ]:
# Cell 5: Loss Function & Optimizer
import auraloss

# L1 time-domain loss
l1_loss = nn.L1Loss()

# Multi-Resolution STFT Loss (from auraloss)
mrstft_loss = auraloss.freq.MultiResolutionSTFTLoss(
    fft_sizes=[512, 1024, 2048],
    hop_sizes=[128, 256, 512],
    win_lengths=[512, 1024, 2048],
    n_bins=128,
    sample_rate=44100
)

def hybrid_loss(pred, target):
    """pred: (B, 2, 2, N), target: (B, 2, 2, N)"""
    # Reshape to (B*2, 2, N) for per-source losses
    B = pred.shape[0]
    pred = pred.view(B * 2, 2, -1)
    target = target.view(B * 2, 2, -1)
    
    l1 = l1_loss(pred, target)
    mrstft = mrstft_loss(pred, target)
    return 0.5 * l1 + 0.5 * mrstft

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)

print("Loss & optimizer configured")

In [ ]:
# Cell 6: Training Loop & Checkpointing
import time
from pathlib import Path
import matplotlib.pyplot as plt

checkpoint_dir = Path('/content/drive/MyDrive/checkpoints')
checkpoint_dir.mkdir(parents=True, exist_ok=True)

num_epochs = 5
train_losses = []

for epoch in range(1, num_epochs + 1):
    model.train()
    epoch_loss = 0.0
    start_time = time.time()
    
    for batch_idx, (mix, targets) in enumerate(dataloader):
        mix = mix.to(device)
        targets = targets.to(device)  # (B, 2, 2, N)
        
        optimizer.zero_grad()
        pred = model(mix)  # (B, 2, 2, N)
        loss = hybrid_loss(pred, targets)
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
        if batch_idx % 50 == 0:
            print(f"  Epoch {epoch} [{batch_idx}/{len(dataloader)}] loss={loss.item():.4f}")
    
    avg_loss = epoch_loss / len(dataloader)
    train_losses.append(avg_loss)
    epoch_time = time.time() - start_time
    
    scheduler.step()
    
    # Save checkpoint
    ckpt_path = checkpoint_dir / f"hdemucs_sanidha_phase1_epoch{epoch}.pt"
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'loss': avg_loss,
        'sources': model.sources,
    }, ckpt_path)
    
    print(f"Epoch {epoch}/{num_epochs} | avg_loss={avg_loss:.4f} | time={epoch_time:.1f}s | saved: {ckpt_path}")

# Plot loss curve
plt.figure(figsize=(8, 4))
plt.plot(range(1, num_epochs + 1), train_losses, 'o-')
plt.xlabel('Epoch')
plt.ylabel('Hybrid Loss')
plt.title('Phase 1 Training Loss (Frozen Encoder)')
plt.grid(True)
plt.show()

print(f"Training complete. Best loss: {min(train_losses):.4f}")